# Adaptação Multimodal do SLMs-CoT — ScienceQA no Google Colab A100 (80GB)

**Par Professor–Aluno:** `Qwen/Qwen2.5-VL-7B-Instruct` (bf16) $\to$ `Qwen/Qwen2.5-VL-3B-Instruct` (bf16)
**Dataset:** ScienceQA (`derek-thomas/ScienceQA`, ~21k exemplos com e sem imagem)
**Objetivo:** Comparação FKL vs RKL vs CE na destilação de raciocínio multimodal com análise segmentada por percepção visual.

---

## Célula 1 — Setup: GPU A100, Clone e Dependências
Verifica a VRAM, monta o Google Drive para persistência e instala as bibliotecas necessárias (`transformers>=4.49.0`, `torchvision`, `qwen-vl-utils`).

In [ ]:
# Importa e roda o setup
import sys, os
if not os.path.isdir('/content/SLMs-CoT'):
    !git clone -b kd-ablations-reweighting https://github.com/Mavitu56/SLMs-CoT.git /content/SLMs-CoT
%cd /content/SLMs-CoT
sys.path.insert(0, '/content/SLMs-CoT')

from scripts.colab_scienceqa_multimodal import cell1_setup
cell1_setup()

## Célula 2 — Verificação dos Modelos e Vocabulário
Confirma o compartilhamento de vocabulário e o alinhamento da dimensão do `lm_head` (7B: 152.064 vs 3B: 151.936).

In [ ]:
from scripts.colab_scienceqa_multimodal import cell2_verify_models
cell2_verify_models()

## Célula 3 — Geração do CoT Piloto (100 exemplos)
Gera uma amostra rápida (~5-10 min) para inspecionar as justificativas e o formato `#### [LETRA]` gerados pelo professor.

In [ ]:
from scripts.colab_scienceqa_multimodal import cell3_generate_pilot_cot
cell3_generate_pilot_cot()

## Célula 4 — Geração Completa do CoT Off-Policy (~12.7k train + 4.2k test)
Executa a geração completa do professor em bf16 com escrita incremental no Google Drive (resumível após quedas de conexão).

In [ ]:
from scripts.colab_scienceqa_multimodal import cell4_generate_full_cot
cell4_generate_full_cot()

## Célula 5 — Smoke Test de Treino (Micro-Overfit)
Roda 1 época com 16 exemplos para assegurar que os tensores visuais, backward pass e gradient checkpointing funcionam sem OOM.

In [ ]:
from scripts.colab_scienceqa_multimodal import cell5_smoke_test_training
cell5_smoke_test_training()

## Célula 6 — Sweep Completo de Treinamento (8 Configurações)
Treina o aluno nas 8 condições: CE baseline (CoT e humano), FKL (T=1, 2, 4) e RKL (T=1, 2, 4). Checkpoints e logs salvos no Google Drive.

In [ ]:
from scripts.colab_scienceqa_multimodal import cell6_run_full_sweep
cell6_run_full_sweep()

## Célula 7 — Avaliação de Acurácia de Múltipla Escolha
Mede a acurácia no split de teste (overall, com imagem e sem imagem).

In [ ]:
from scripts.colab_scienceqa_multimodal import cell7_evaluate_accuracy
cell7_evaluate_accuracy()

## Célula 8 — Avaliação Probabilística e Calibração
Calcula ECE, entropia $H_R$ e $H_A$, razão $\rho = H_R / H_A$, e divergência KL entre professor e aluno, decomposta por presença de imagem.

In [ ]:
from scripts.colab_scienceqa_multimodal import cell8_evaluate_probabilistic
cell8_evaluate_probabilistic()